In [16]:
import scanpy as sc
import numpy as np
import os

input_dir = "./data_output/cortex_h5ad"
files = [f for f in os.listdir(input_dir) if f.endswith(".h5ad")]

N_HVG = 2000
N_PCS = 10

for f in files:
    in_path = os.path.join(input_dir, f)
    out_path = os.path.join(input_dir, f.replace(".h5ad", f".hvg{N_HVG}.pca{N_PCS}.h5ad"))

    print("Processing:", f)
    adata = sc.read_h5ad(in_path)

    # --- 基本检查 ---
    if adata.n_vars < N_HVG:
        print(f"  ⚠️ n_vars={adata.n_vars} < {N_HVG}, skipping")
        continue

    # --- 1) 预处理：normalize + log1p ---
    # 如果你的 X 不是 raw counts（例如已经是log），这里会改变数值；
    # 但你说要“自己再做一遍”，就统一重做。
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # --- 2) 找高变基因前2000 ---
    # seurat_v3 常见且效果好；需要较新版本 scanpy
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_HVG,
        flavor="seurat",
        subset=True  # 直接把 adata 只保留HVG
    )

    # --- 3) scale（PCA前常用） ---
    # max_value 防止极端值影响
    sc.pp.scale(adata, max_value=10)

    # --- 4) PCA：只算前10维 ---
    sc.tl.pca(adata, n_comps=N_PCS, svd_solver="arpack")

    # 强制确保只保留10维（以防某些版本/设置带来额外维度）
    Xpca = np.asarray(adata.obsm["X_pca"])[:, :N_PCS].copy()
    adata.obsm["X_pca"] = Xpca
    adata.obsm["PCA"] = Xpca  # 兼容老key

    # --- 5) 写出 ---
    adata.write_h5ad(out_path)
    print("  saved:", out_path)
    print("  X_pca shape:", adata.obsm["X_pca"].shape)
    print()

Processing: F1_NSS_vs_B73_SS.h5ad
  saved: ./data_output/cortex_h5ad/F1_NSS_vs_B73_SS.hvg2000.pca10.h5ad
  X_pca shape: (13659, 10)

Processing: F1_TL_vs_P1_TL.h5ad
  saved: ./data_output/cortex_h5ad/F1_TL_vs_P1_TL.hvg2000.pca10.h5ad
  X_pca shape: (8594, 10)

Processing: F1_SS_vs_B73_SS.h5ad
  saved: ./data_output/cortex_h5ad/F1_SS_vs_B73_SS.hvg2000.pca10.h5ad
  X_pca shape: (11408, 10)

Processing: F1_NSS_vs_P1_NSS.h5ad
  saved: ./data_output/cortex_h5ad/F1_NSS_vs_P1_NSS.hvg2000.pca10.h5ad
  X_pca shape: (10667, 10)

Processing: F1_TL_vs_B73_SS.h5ad
  saved: ./data_output/cortex_h5ad/F1_TL_vs_B73_SS.hvg2000.pca10.h5ad
  X_pca shape: (11383, 10)

Processing: F1_SS_vs_P1_SS.h5ad
  saved: ./data_output/cortex_h5ad/F1_SS_vs_P1_SS.hvg2000.pca10.h5ad
  X_pca shape: (6911, 10)



In [26]:
result_dir = "./data_output/mmd_result"

data = {}

for folder in sorted(os.listdir(result_dir)):

    folder_path = os.path.join(result_dir, folder)
    
    if not os.path.isdir(folder_path):
        continue
    
    npy_path = os.path.join(
        folder_path,
        "mmd_distances_group_mmd_celltype.npy"
    )
    
    if not os.path.exists(npy_path):
        print("Missing:", npy_path)
        continue
    
    arr = np.load(npy_path)
    
    print(folder, "shape:", arr.shape)
    
    # 取[:,:,1]
    mmd = arr[:, :, 0]
    
    # 展平成一列
    mmd_flat = mmd.flatten()
    
    data[folder] = mmd_flat


df = pd.DataFrame(data)

out_path = "./data_output/mmd_result/mmd_summary.csv"

df.to_csv(out_path, index=False)

print("Saved:", out_path)

F1_NSS_vs_B73_SS shape: (1, 100, 3)
F1_NSS_vs_P1_NSS shape: (1, 100, 3)
F1_SS_vs_B73_SS shape: (1, 100, 3)
F1_SS_vs_P1_SS shape: (1, 100, 3)
F1_TL_vs_B73_SS shape: (1, 100, 3)
F1_TL_vs_P1_TL shape: (1, 100, 3)
Saved: ./data_output/mmd_result/mmd_summary.csv


In [21]:
data_test = np.load("data_output/mmd_result/F1_NSS_vs_B73_SS/mmd_distances_group_mmd_celltype.npy")

In [25]:
data_test[:,:,0]

array([[0.05893004, 0.05390066, 0.05173743, 0.05482745, 0.05729371,
        0.06029344, 0.05718863, 0.05405533, 0.05167359, 0.04220963,
        0.0585146 , 0.06618965, 0.06497264, 0.06429988, 0.05952644,
        0.05019218, 0.0557546 , 0.05589396, 0.06087065, 0.05627155,
        0.06546855, 0.06583697, 0.06224668, 0.04889321, 0.03435761,
        0.04755008, 0.0587579 , 0.06165975, 0.05787742, 0.0515011 ,
        0.04991114, 0.05265027, 0.05143547, 0.05010521, 0.05264622,
        0.0604291 , 0.04362255, 0.05455446, 0.07237762, 0.0522033 ,
        0.048509  , 0.04843003, 0.03793949, 0.05901217, 0.05654848,
        0.04196924, 0.05264711, 0.05461174, 0.06021744, 0.05861807,
        0.06754935, 0.05031693, 0.05940211, 0.06012505, 0.04336137,
        0.06857067, 0.05340415, 0.05616534, 0.06676477, 0.06178218,
        0.06553251, 0.04951197, 0.0586341 , 0.05988365, 0.06595641,
        0.0416652 , 0.06179738, 0.0589065 , 0.05557293, 0.05629897,
        0.0473187 , 0.054869  , 0.06408662, 0.07